# eg9 (v2) — `listen`: 対象更新 → event(pull 先行・pump の要否をこの notebook で判断)

v1 の eg9(`eg9_listner.ipynb`)は `listen('a')` で対象 `a` の更新を購読し、browser が `getValueString('a')` を push して `comm.shared_objects['a']` を mirror、`add_shared_listener` の callback で ipywidgets の Label を生更新した。
v2 では applet の add/update listener は mount 時に一度だけ配線され、event は server の郵便箱(document 単位の log)に積まれる。kernel は `events()` で pull し、`listen(cb, label=)` の callback は pull の時に発火する(裁定 (ii) 09-09: pull 先行)。
この notebook は v1 の場面を順に再生し、(a) pull で v1 の意味論が再現できるか、(b) pump(kernel 内 thread)の代替 = blocking pull(`wait_update`)、(c) pump の陽性統制(ad hoc thread)が何をもたらすか、を同じ計器で測る。数値は RECORD に写す。

In [ ]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
from ggblab import GeoGebra
g = GeoGebra(appName='suite', showAlgebraInput=True); g

## 1. v1 case 1 逐語: 代数命令(`l1 = {Intersect(c1, c2)}`・`a = Length(l1)`)

In [ ]:
print(g.command("O = (0, 0)", "c1 = Circle(O, 1)", "A = (1, 0)", "c2 = Circle(A, 1)", "l1 = {Intersect(c1, c2)}", "a = Length(l1)"))
print("a =", g.value("a"))            # v1: shared_objects['a'] == 'a = 2'

## 2. pull: mount 以後の event を引く(v1 の `comm.logs` に相当)

In [ ]:
evs = g.events()
print(len(evs), "events since mount:", [(e["type"], e["label"]) for e in evs][:12])

## 3. v1 `listen('a')` の再生: 対象 `a` だけの callback(pull 時に発火)+ mirror 値は `value('a')`

In [ ]:
seen = []
g.listen(lambda e: seen.append((e["type"], e["label"], g.value("a"))), label="a")
print(g.command("A = (1.5, 0)"))
print("pull ->", [(e["type"], e["label"]) for e in g.events()])
print("seen:", seen)
print("a =", g.value("a"))

## 4. 操作の再生: applet 上で A をドラッグしてから次の cell を走らせる

(自動測定では Playwright が `setCoords('A', x, 0)` 20 手でドラッグを代行する。kernel は何も命じていないので、event は browser 側の操作だけから出る)

In [ ]:
time.sleep(8)                        # 操作の時間(授業では「今ドラッグ」・自動測定ではこの間に 20 手)
t0 = time.time(); evs = g.events(); dt = time.time() - t0
ups = [e for e in evs if e["type"] == "update" and e["label"] == "A"]
print("pulled", len(evs), "events in", round(dt, 3), "s; A updates", len(ups), "; a =", g.value("a"), "; seen(a)", len(seen))

## 5. pump の代替 = blocking pull: 次の操作まで cell が待つ(`wait_update`・kernel に thread なし)

In [ ]:
print("waiting for the next update of A (<= 20 s) ..."); t0 = time.time()
e = g.wait_update("A", timeout=20)
print("woke after", round(time.time() - t0, 2), "s:", e, "; a =", g.value("a"))

## 6. pump の陽性統制: ad hoc thread が cell の外で event を集める(v1 の callback 生更新に相当)

In [ ]:
import threading
box, stop = [], threading.Event()
def pump():
    while not stop.is_set():
        for e in g.events(wait=5):
            box.append(e)
th = threading.Thread(target=pump, daemon=True); th.start()
time.sleep(8)                        # 操作の時間(自動測定: 20 手)
stop.set(); th.join(timeout=12)
print("pump collected", len(box), "events while the cell only slept; A updates", sum(1 for e in box if e["label"] == "A"))

## 7. pump の副作用の計測: cell が終わった後に thread が print すると、その出力はどこへ行くか

In [ ]:
stop2 = threading.Event()
def pump2():
    n = 0
    while not stop2.is_set():
        for e in g.events(wait=5):
            n += 1
            if n == 1: print("pump2: first event", (e["type"], e["label"]), "— printed from the thread AFTER this cell finished")
threading.Thread(target=pump2, daemon=True).start(); print("pump2 started; move A in the applet now")

## 8. 判断(数値は RECORD へ・裁定は先生)

- (a) pull だけで v1 の意味論(購読・mirror 値・callback)は再現できるか → §2–§4
- (b) 「操作に反応する」は blocking pull で足りるか → §5
- (c) pump が要るのはどの場面か・何を壊すか → §6–§7(出力の行き先)

In [ ]:
time.sleep(6); stop2.set(); print("DONE")